In [ ]:
# Repo bootstrap: notebooks use paths relative to the repo root and import from src/
import os, sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "pipeline.py").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2
import shutil
import torch.nn as nn
import os
import pandas as pd
import random

from PatchDataset import load_dataset
from torch.utils.data import DataLoader,Subset,Dataset
from torchvision import datasets, transforms
from PIL import Image
from tqdm import tqdm
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.metrics import (
    roc_curve, auc,
    precision_recall_curve, average_precision_score
)
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report, confusion_matrix
from transformers import CLIPProcessor, CLIPModel

from PatchDataset import load_dataset,load_augmented_dataset
from pipeline import set_seed,predict_probs, eval_clip,compute_binary_metrics

PATCH_SIZE = 224
BATCH_SIZE = 32
SEED = 42
set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

PATCHES_ROOT = "patches_dataset/patches_v3_seed42_pad1.6_iouth0.09"
traincsv_path = os.path.join(PATCHES_ROOT, "metadata.csv")
testcsv_path = "patches_dataset/test_patches_v3/test_metadata.csv"

# loading the dataset
traindf = pd.read_csv(traincsv_path)
traindf["patch_id"] = traindf["patch_id"].astype(int)
traindf["label"] = traindf["label"].astype(int)
traindf["type"] = traindf["type"].astype(str)

testdf = pd.read_csv(testcsv_path)
testdf["patch_id"] = testdf["patch_id"].astype(int)
testdf["label"] = testdf["label"].astype(int)
testdf["type"] = testdf["type"].astype(str)

train_dataset,val_dataset,train_loader,val_loader,test_dataset,test_loader = load_dataset(traindf,testdf,PATCHES_ROOT,BATCH_SIZE)

for xb,yb in train_loader:
    print(xb.shape,yb.shape,min(yb),max(yb))
    break

AUG_ROOT = "patches_dataset/patches_v3_train_aug"
PATH_AUG_METADATA = os.path.join(AUG_ROOT, "metadata.csv")

aug_train_dataset,aug_val_dataset,aug_train_loader,aug_val_loader,aug_test_dataset,aug_test_loader = load_augmented_dataset(
    PATCHES_ROOT,
    AUG_ROOT,
    PATH_AUG_METADATA,
    testcsv_path,
    BATCH_SIZE
)

for xb, yb in aug_train_loader:
    print(xb.shape, yb.shape, min(yb), max(yb))
    break


cpu
train/val: 15858 3965
torch.Size([32, 3, 224, 224]) torch.Size([32]) tensor(0) tensor(1)
train_aug/test: 25372 3965
torch.Size([32, 3, 224, 224]) torch.Size([32]) tensor(0) tensor(1)


In [2]:
import clip
clip.available_models()  

['RN50',
 'RN101',
 'RN50x4',
 'RN50x16',
 'RN50x64',
 'ViT-B/32',
 'ViT-B/16',
 'ViT-L/14',
 'ViT-L/14@336px']

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from tqdm import tqdm
import numpy as np
import clip
from clip.simple_tokenizer import SimpleTokenizer as _Tokenizer

_tokenizer = _Tokenizer()

# ─────────────────────────────────────────────
# Config — replaces Dassl's cfg object
# ─────────────────────────────────────────────
class CoOpConfig:
    def __init__(
        self,
        n_ctx=16,                    # number of learnable context tokens
        ctx_init="",                 # e.g. "a photo of a" to init with text, "" for random
        class_token_position="end",  # "end", "middle", or "front"
        csc=False,                   # class-specific context (one ctx vector per class)
        prec="fp32",                 # "fp32", "fp16", or "amp"
    ):
        self.n_ctx = n_ctx
        self.ctx_init = ctx_init
        self.class_token_position = class_token_position
        self.csc = csc
        self.prec = prec


# ─────────────────────────────────────────────
# Text Encoder — CLIP's transformer, no changes
# ─────────────────────────────────────────────
class TextEncoder(nn.Module):
    def __init__(self, clip_model):
        super().__init__()
        self.transformer = clip_model.transformer
        self.positional_embedding = clip_model.positional_embedding
        self.ln_final = clip_model.ln_final
        self.text_projection = clip_model.text_projection
        self.dtype = clip_model.dtype

    def forward(self, prompts, tokenized_prompts):
        x = prompts + self.positional_embedding.type(self.dtype)
        x = x.permute(1, 0, 2)   # NLD -> LND
        x = self.transformer(x)
        x = x.permute(1, 0, 2)   # LND -> NLD
        x = self.ln_final(x).type(self.dtype)
        # take the eot (end-of-text) token feature
        x = x[torch.arange(x.shape[0]), tokenized_prompts.argmax(dim=-1)]
        x = x @ self.text_projection
        return x


# ─────────────────────────────────────────────
# Prompt Learner — the core of CoOp
# ─────────────────────────────────────────────
class PromptLearner(nn.Module):
    def __init__(self, cfg: CoOpConfig, classnames: list[str], clip_model):
        super().__init__()
        n_cls = len(classnames)
        n_ctx = cfg.n_ctx
        dtype = clip_model.dtype
        ctx_dim = clip_model.ln_final.weight.shape[0]  # 512 for ViT-B/32

        # ── Initialize context vectors ──
        if cfg.ctx_init:
            # init from a text string, e.g. "a photo of a"
            ctx_init = cfg.ctx_init.replace("_", " ")
            n_ctx = len(ctx_init.split(" "))
            prompt = clip.tokenize(ctx_init)
            with torch.no_grad():
                embedding = clip_model.token_embedding(prompt).type(dtype)
            ctx_vectors = embedding[0, 1: 1 + n_ctx, :]
            prompt_prefix = ctx_init
        else:
            # random init
            if cfg.csc:
                # one context vector per class
                ctx_vectors = torch.empty(n_cls, n_ctx, ctx_dim, dtype=dtype)
            else:
                # shared context across all classes
                ctx_vectors = torch.empty(n_ctx, ctx_dim, dtype=dtype)
            nn.init.normal_(ctx_vectors, std=0.02)
            prompt_prefix = " ".join(["X"] * n_ctx)

        print(f'Initial context: "{prompt_prefix}"')
        print(f"Number of context tokens: {n_ctx}")

        self.ctx = nn.Parameter(ctx_vectors)  # this is what gets optimized

        # ── Tokenize class names and build fixed prefix/suffix embeddings ──
        classnames = [name.replace("_", " ") for name in classnames]
        name_lens = [len(_tokenizer.encode(name)) for name in classnames]
        prompts = [prompt_prefix + " " + name + "." for name in classnames]

        tokenized_prompts = torch.cat([clip.tokenize(p) for p in prompts])
        with torch.no_grad():
            embedding = clip_model.token_embedding(tokenized_prompts).type(dtype)

        # SOS token (index 0), then ctx slots, then class name + EOS
        self.register_buffer("token_prefix", embedding[:, :1, :])         # (n_cls, 1, dim)
        self.register_buffer("token_suffix", embedding[:, 1 + n_ctx:, :]) # (n_cls, *, dim)

        self.n_cls = n_cls
        self.n_ctx = n_ctx
        self.tokenized_prompts = tokenized_prompts
        self.name_lens = name_lens
        self.class_token_position = cfg.class_token_position

    def forward(self):
        ctx = self.ctx
        if ctx.dim() == 2:
            # shared context: expand to all classes
            ctx = ctx.unsqueeze(0).expand(self.n_cls, -1, -1)

        prefix = self.token_prefix
        suffix = self.token_suffix

        if self.class_token_position == "end":
            # [SOS] [ctx...] [class] [EOS]
            prompts = torch.cat([prefix, ctx, suffix], dim=1)

        elif self.class_token_position == "middle":
            # [SOS] [ctx_half1] [class] [ctx_half2] [EOS]
            half = self.n_ctx // 2
            prompts = []
            for i in range(self.n_cls):
                name_len = self.name_lens[i]
                prompts.append(torch.cat([
                    prefix[i:i+1],
                    ctx[i:i+1, :half],
                    suffix[i:i+1, :name_len],
                    ctx[i:i+1, half:],
                    suffix[i:i+1, name_len:],
                ], dim=1))
            prompts = torch.cat(prompts, dim=0)

        elif self.class_token_position == "front":
            # [SOS] [class] [ctx...] [EOS]
            prompts = []
            for i in range(self.n_cls):
                name_len = self.name_lens[i]
                prompts.append(torch.cat([
                    prefix[i:i+1],
                    suffix[i:i+1, :name_len],
                    ctx[i:i+1],
                    suffix[i:i+1, name_len:],
                ], dim=1))
            prompts = torch.cat(prompts, dim=0)

        else:
            raise ValueError(f"Unknown class_token_position: {self.class_token_position}")

        return prompts


# ─────────────────────────────────────────────
# CustomCLIP — full model
# ─────────────────────────────────────────────
class CustomCLIP(nn.Module):
    def __init__(self, cfg: CoOpConfig, classnames: list[str], clip_model):
        super().__init__()
        self.prompt_learner = PromptLearner(cfg, classnames, clip_model)
        self.tokenized_prompts = self.prompt_learner.tokenized_prompts
        self.image_encoder = clip_model.visual
        self.text_encoder = TextEncoder(clip_model)
        self.logit_scale = clip_model.logit_scale
        self.dtype = clip_model.dtype

    def forward(self, image):
        image_features = self.image_encoder(image.type(self.dtype))
        prompts = self.prompt_learner()
        text_features = self.text_encoder(prompts, self.tokenized_prompts)

        image_features = F.normalize(image_features, dim=-1)
        text_features = F.normalize(text_features, dim=-1)

        logit_scale = self.logit_scale.exp()
        logits = logit_scale * image_features @ text_features.t()  # (B, n_cls)
        return logits


# ─────────────────────────────────────────────
# Training
# ─────────────────────────────────────────────
def train_coop(
    classnames: list[str],
    train_loader,
    backbone="ViT-B/32",
    num_epochs=10,
    lr=2e-3,
    n_ctx=16,
    ctx_init="a photo of a",
    class_token_position="end",
    prec="fp32",           # use "amp" if you want mixed precision
    device=None,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    cfg = CoOpConfig(
        n_ctx=n_ctx,
        ctx_init=ctx_init,
        class_token_position=class_token_position,
        prec=prec,
    )

    print(f"Loading CLIP backbone: {backbone}")
    clip_model, preprocess = clip.load(backbone, device="cpu")  # load to cpu first

    if prec == "fp32":
        clip_model.float()  # CLIP defaults to fp16

    model = CustomCLIP(cfg, classnames, clip_model).to(device)

    # freeze everything except the prompt learner
    for name, param in model.named_parameters():
        if "prompt_learner" not in name:
            param.requires_grad_(False)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {trainable}")

    optimizer = torch.optim.SGD(model.prompt_learner.parameters(), lr=lr)
    scaler = GradScaler() if prec == "amp" else None

    model.train()
    for epoch in range(num_epochs):
        total_loss = 0.0
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            images = images.to(device)
            labels = labels.to(device).long()

            if prec == "amp":
                with autocast():
                    logits = model(images)
                    loss = F.cross_entropy(logits, labels)
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(images)
                loss = F.cross_entropy(logits, labels)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.prompt_learner.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1} — Loss: {total_loss / len(train_loader):.4f}")

    return model


# ─────────────────────────────────────────────
# Evaluation
# ─────────────────────────────────────────────
def eval_coop(model, test_loader, device):
    model.eval()
    all_probs, all_labels = [], []

    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Evaluating"):
            images = images.to(device)
            logits = model(images)                          # (B, n_cls)
            probs = torch.softmax(logits, dim=-1)[:, 1]    # P(tree) if binary
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.numpy())

    return np.array(all_probs), np.array(all_labels)

In [5]:
# your class names — adapt to your actual tree classes
classnames = ["non-tree", "tree"]

device = "cuda" if torch.cuda.is_available() else "cpu"

coop_model = train_coop(
    classnames=classnames,
    train_loader=train_loader,
    backbone="ViT-B/32",
    num_epochs=10,
    lr=2e-3,
    n_ctx=16,
    ctx_init="a photo of a",   # or "" for random init
    prec="fp32",
    device=device,
)

probs, labels = eval_coop(coop_model, test_loader, device)

Loading CLIP backbone: ViT-B/32


100%|███████████████████████████████████████| 338M/338M [00:13<00:00, 26.5MiB/s]


Initial context: "a photo of a"
Number of context tokens: 4
Trainable parameters: 2048


Epoch 1/10:   1%|          | 6/496 [00:12<16:58,  2.08s/it]


KeyboardInterrupt: 